## Importing libraries

In [1]:
import numpy as np
import pandas as pd

## Importing data

In [2]:
string_cols = [
    'PSEUDO_PAZ',
    'STRUTTURA_COD',
    'SESSO_COD',
    'PROVENIENZA_COD',
    'ISTRUZIONE_COD',
    'REGIME_RICOVERO_COD',
]
dtype = {col: 'string' for col in string_cols}


# int_cols = [
#     '',
# ]
# dtype.update({col: 'Int64' for col in int_cols})

# ***_cols = [
#     '',
# ]
# dtype.update({col: 'category' for col in ***_cols})

In [3]:
df = pd.read_excel('../raw_data/Estrazioni_iconsulting.xlsx', sheet_name='Ricoveri',
    dtype_backend='numpy_nullable', # For allowing the `string` dtype
    dtype=dtype
)

# Make the index start from 2 (as the original excel)
df.index = df.index + 2

## Parsing and reformatting

Place NaT values instead of '9999-09-09 00:00:00' in columns:

In [4]:
to_NaT_values = [
    'DATA_PRENOTAZIONE',
    'DATA_CHIUSURA_SDO'
]
df[to_NaT_values] = (df[to_NaT_values].replace(
    pd.Timestamp('9999-09-09 00:00:00'), pd.NaT
))

Place NA values instead of single whitespaces in columns:

In [5]:
to_NA_values = [
    'PRIORITA_PRENOTAZIONE_COD',
    'STRUTTURA_PROVENIENZA_COD',
    'STRUTTURA_DESTINAZIONE_COD'
]
df[to_NA_values] = (df[to_NA_values].replace(' ', pd.NA))

## Final checks

In [6]:
df

,PSEUDO_PAZ,ANNO,DATA_APERTURA,DATA_CHIUSURA,MODALITA_DIMISSIONE_COD,STRUTTURA_COD,TIPO_ATTIVITA_COD,SESSO_COD,PROVENIENZA_COD,ISTRUZIONE_COD,...,'S03','S04','INTERVENTO_P','INTERVENTO_S01','INTERVENTO_S02','INTERVENTO_S03','INTERVENTO_S04','INTERVENTO_S',COMPILAZIONE_CHECKLIST_COD,"LISTAGG(C.DATA_INTERVENTO,',')OVER(PARTITIONBYA.NOSOLOGICO,A.ANNO,A.STRUTTURA_COD)"
2,20525758880092952616734,2022,2022-08-01 15:47:00,2022-08-10 14:30:00,2,042006,RI,1,02,9,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"01-AGO-22,01-AGO-22,01-AGO-22,01-AGO-22"
3,2719961031659409949680,2020,2020-12-08 16:09:00,2020-12-15 16:30:00,2,042006,RI,1,05,9,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"08-DIC-20,08-DIC-20,08-DIC-20,08-DIC-20,08-DIC..."
4,30961298531313452439580,2022,2022-01-17 20:10:00,2022-01-21 13:10:00,2,042006,RI,1,02,9,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"17-GEN-22,17-GEN-22,17-GEN-22,17-GEN-22"
5,2719961031659409949680,2020,2020-12-08 16:09:00,2020-12-15 16:30:00,2,042006,RI,1,05,9,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"08-DIC-20,08-DIC-20,08-DIC-20,08-DIC-20,08-DIC..."
6,5712560221500663468891,2001,2001-01-17 08:00:00,2001-06-08 00:00:00,2,042006,RI,1,1,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1380,5443382870799330932690,2022,2022-08-29 03:44:00,2022-09-21 15:00:00,8,042001,RI,1,05,9,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"29-AGO-22,30-AGO-22,30-AGO-22,30-AGO-22,30-AGO..."
1381,8688775490799534962197,2018,2018-05-22 08:00:00,2018-05-29 13:00:00,8,042006,RI,1,03,9,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"25-MAG-18,23-MAG-18"
1382,13280116591091443765758,2017,2017-03-06 08:15:00,2017-03-15 12:00:00,8,042006,RI,1,04,2,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"09-MAR-17,09-MAR-17,09-MAR-17,09-MAR-17,09-MAR..."
1383,2462337189602073236450,2022,2022-03-29 13:09:00,2022-04-26 18:20:00,9,042010,RI,2,05,9,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,30-MAR-22


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1383 entries, 2 to 1384
Data columns (total 46 columns):
 #   Column                                                                              Non-Null Count  Dtype         
---  ------                                                                              --------------  -----         
 0   PSEUDO_PAZ                                                                          1383 non-null   string        
 1   ANNO                                                                                1383 non-null   Int64         
 2   DATA_APERTURA                                                                       1383 non-null   datetime64[us]
 3   DATA_CHIUSURA                                                                       1383 non-null   datetime64[us]
 4   MODALITA_DIMISSIONE_COD                                                             1383 non-null   string        
 5   STRUTTURA_COD                                              

## Export

In [ ]:
df.to_parquet("../raw_data/parquets/dati_estrazioni_ricoveri.parquet"
    # , compression="gzip"
)

## Inconsistency between `PSEUDO_PAZ` and other columns

### Proof of the inconsistency: see results for code structure

`PROVENIENZA_COD`, `ISTRUZIONE_COD` are inconsistent

In [9]:
# (df.groupby('PSEUDO_PAZ')['SESSO_COD'].nunique() == 1).all()
# (df.groupby('PSEUDO_PAZ')['PROVENIENZA_COD'].nunique() == 1).all()
# (df.groupby('PSEUDO_PAZ')['ISTRUZIONE_COD'].nunique() == 1).all()

### See which `PSEUDO_PAZ` have multiple characteristics that should be unique

In [10]:
# incons = df.groupby('PSEUDO_PAZ')[''].nunique()
# incons = incons[incons > 1].sort_values(ascending=False)
# incons